# Feature table – step 2: features of every recording

This notebook measures the four features of every recording of the 13 metadata files saved
by notebook 08 and saves one feature table per dataset and subset.

| feature | column |
|---|---|
| duration | `duration_s` (from the file header) |
| silence | `leading_silence_s_-30dB`, `trailing_silence_s_-30dB` |
| loudness | `average_loudness_dbfs` (RMS of the whole recording, silence included) |
| bandwidth | `bandwidth_hz` |

The column `error` is empty, or holds the error message of a file that could not be read
(its features are then empty). All settings (threshold, frame length, ...) are in
`src/xai_audio_deepfake_shortcuts_audit/features/config.py`.

**How to use it**

- Run notebook 08 first: this notebook reads its files in `outputs/08_feature_table_output/metadata/`.
- Run the cells from top to bottom. Step 2 measures about one million recordings and takes
  a long time (notebook 09a estimates how long). It saves its results every few thousand recordings: if it is stopped, run it
  again and it continues where it stopped.
- The notebook only **reads** `datasets/` – nothing there is changed.

**Output files**

In `outputs/09_feature_extraction_output/` (not committed to git):

- `features/features_<dataset>_<subset>_-30dB.csv` – the feature tables (step 2).

Read them with `read_feature_table`, so that the metadata stay text and empty features
become `NaN`.

**Steps**

1. Setup – paths and settings
2. Features of all recordings
3. Files that could not be read
4. Check of the measurements: silence and bandwidth in ASVspoof 2021

### Step 1 – Setup: paths and settings

**What it does:** Finds the project folder, sets the folder of the metadata files (from
notebook 08), the output folders and the number of processes that measure recordings at
the same time (all processor cores but one). It also lists the 13 metadata files in the
order in which they will be processed – the largest dataset (ASVspoof 2021 DF) last – and
shows how many recordings each one has.

**Why:** All the next cells use these paths and settings. The threshold from `config.py`
is written in the names of the output files, so a run with another threshold does not
overwrite them.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from xai_audio_deepfake_shortcuts_audit.features.config import SAMPLE_RATE_HZ, THRESHOLD_DB
from xai_audio_deepfake_shortcuts_audit.features.read_feature_table import read_feature_table


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("pyproject.toml not found")


REPO_ROOT = find_repo_root(Path.cwd())
METADATA_DIR = REPO_ROOT / "outputs" / "08_feature_table_output" / "metadata"
NOTEBOOK_NAME = "09_feature_extraction"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
FEATURES_DIR = OUTPUTS_DIR / "features"

SUFFIX = f"{THRESHOLD_DB:+.0f}dB"  # e.g. -30dB, written in the names of the output files
WORKERS = max(1, (os.cpu_count() or 2) - 1)  # all processor cores but one
CHUNK_SIZE = 5000  # recordings measured between two saves

# datasets and subsets in the order of processing; the largest dataset last
SUBSETS = {
    "01_asvspoof_2019": ["train", "dev", "eval"],
    "04_in_the_wild": ["all"],
    "02_asvspoof_2021_la": ["progress", "eval", "hidden"],
    "05_sonics": ["train", "valid", "test"],
    "03_asvspoof_2021_df": ["progress", "eval", "hidden"],
}

files = pd.DataFrame(
    [
        {"dataset": dataset, "subset": subset, "metadata_file": f"metadata_{dataset}_{subset}.csv"}
        for dataset, subsets in SUBSETS.items()
        for subset in subsets
    ]
)
files["exists"] = [(METADATA_DIR / name).exists() for name in files["metadata_file"]]
files["rows"] = [
    len(read_feature_table(METADATA_DIR / name)) if exists else 0
    for name, exists in zip(files["metadata_file"], files["exists"], strict=True)
]
print("Metadata folder:", METADATA_DIR.relative_to(REPO_ROOT))
print("Output folder:", OUTPUTS_DIR.relative_to(REPO_ROOT))
print("Threshold:", SUFFIX, "| sample rate:", SAMPLE_RATE_HZ, "Hz | processes:", WORKERS)
display(files)
print(f"All recordings: {files['rows'].sum():,}")

### Step 2 – Features of all recordings

**What it does:** For every metadata file (in the order of step 1) `build_feature_table`
measures the features of every recording with `WORKERS` processes and appends the rows to
`features/features_<dataset>_<subset>_-30dB.csv` every `CHUNK_SIZE` recordings. During the run
it prints the progress and the estimated time left. At the end it shows one row per file:
recordings in the file, recordings measured in this run, files with an error and time.

**Why:** This is the feature table. If the run stops (closed laptop, kernel restart), run
this cell again: the recordings already in the files are skipped and the run continues where
it stopped. A finished file is not measured again.

In [ ]:
from xai_audio_deepfake_shortcuts_audit.features.build_feature_table import build_feature_table

summaries = []
for dataset, subset, name in zip(
    files["dataset"], files["subset"], files["metadata_file"], strict=True
):
    metadata = read_feature_table(METADATA_DIR / name)
    out_path = FEATURES_DIR / f"features_{dataset}_{subset}_{SUFFIX}.csv"
    summary = build_feature_table(metadata, REPO_ROOT, out_path, WORKERS, CHUNK_SIZE)
    summaries.append({"dataset": dataset, "subset": subset, **summary})

summary = pd.DataFrame(summaries)
summary["out_path"] = [path.relative_to(REPO_ROOT).as_posix() for path in summary["out_path"]]
display(summary)
print(f"All recordings: {summary['rows'].sum():,} | with an error: {summary['errors'].sum():,}")

### Step 3 – Files that could not be read

**What it does:** Reads all feature tables and shows the recordings with a non-empty
`error` column (at most 20 rows) and the number of such recordings per dataset and subset.

**Why:** The features of these recordings are empty, so they are left out later. The error
message tells why the file could not be read.

In [ ]:
tables = [
    read_feature_table(path) for path in sorted(FEATURES_DIR.glob(f"features_*_{SUFFIX}.csv"))
]
features = pd.concat(tables, ignore_index=True)
failed = features[features["error"] != ""]

print(f"Recordings: {len(features):,} | with an error: {len(failed):,}")
if len(failed):
    display(failed.groupby(["dataset", "subset"]).size().to_frame("errors"))
    display(failed[["dataset", "subset", "file_id", "error"]].head(20))